# Phase 4: Integration 1: Unified Data Table

## Overview
Merge outputs from all analysis phases into unified master tables at multiple granularity levels.
This is pure data engineering: no interpretation, no task-specific logic.

## Purpose
Downstream integration notebooks (NB02, NB03) load these tables instead of scattered CSVs from individual phases.

## Output Tables
1. **Per-circuit** (N_models x N_conditions x N_replications): functional + structural + representational metrics per circuit
2. **Pairwise** (N_models x N_condition_pairs): transfer, Jaccard, CKA between condition pairs
3. **Model-level** (N_models): aggregated metrics per model
4. **Metric catalog**: registry of all metrics with source phase, file, and granularity

## Data Sources
- Phase 1 (Functional): `full_circuit_data.csv`
- Phase 2 (Structural): `full_structure_data.csv`, `universal_edges.csv`
- Phase 3 (Representational): master CSVs from logit_lens, residual_stream, info_theoretic, transfer
- Phase 5 (Targeted): `scaling_summary.csv` (model-level enrichment)

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# Phase paths
PROJECT_ROOT = Path("LSC_circuit_analysis")
PHASE1_DIR = PROJECT_ROOT / "01_Phase_Functional"
PHASE2_DIR = PROJECT_ROOT / "02_Phase_Structural"
PHASE3_DIR = PROJECT_ROOT / "03_Phase_Representational"
PHASE4_DIR = PROJECT_ROOT / "04_Phase_Integration"
PHASE5_DIR = PROJECT_ROOT / "05_Phase_Targeted"

# Add utils to path
sys.path.insert(0, str(PHASE4_DIR))
from utils.constants import (
    MODELS,
    FREQUENCY_BANDS,
    DRAWS,
    MODEL_CAPACITY,
    BAND_COLORS,
    MODEL_COLORS,
    FREQUENCY_RANK,
)

# Output directories
ANALYSIS_DIR = PHASE4_DIR / "outputs" / "analysis"
VIZ_DIR = PHASE4_DIR / "outputs" / "viz"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Generic condition/replication labels
CONDITIONS = FREQUENCY_BANDS  # ['low', 'medium', 'high', 'very_high']
REPLICATIONS = DRAWS  # ['draw_1', 'draw_2', 'draw_3']

print(f"Models: {MODELS}")
print(f"Conditions: {CONDITIONS}")
print(f"Replications: {REPLICATIONS}")
print(
    f"Expected per-circuit rows: {len(MODELS)} x {len(CONDITIONS)} x {len(REPLICATIONS)} = {len(MODELS) * len(CONDITIONS) * len(REPLICATIONS)}"
)

Models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']
Conditions: ['low', 'medium', 'high', 'very_high']
Replications: ['draw_1', 'draw_2', 'draw_3']
Expected per-circuit rows: 5 x 4 x 3 = 60


## 1. Load Source Data from Each Phase

In [2]:
# --- Phase 1: Functional ---
df_func = pd.read_csv(PHASE1_DIR / "outputs" / "analysis" / "full_circuit_data.csv")
# Keep only frequency bands (exclude control)
df_func = df_func[df_func["band"].isin(CONDITIONS)].copy()
print(f"Phase 1 (Functional): {len(df_func)} rows, columns: {list(df_func.columns)}")

# --- Phase 2: Structural ---
df_struct = pd.read_csv(PHASE2_DIR / "outputs" / "analysis" / "full_structure_data.csv")
df_struct = df_struct[df_struct["band"].isin(CONDITIONS)].copy()
print(f"Phase 2 (Structural): {len(df_struct)} rows")

# Phase 2: Universal edges (model x draw level)
df_univ = pd.read_csv(PHASE2_DIR / "outputs" / "analysis" / "universal_edges.csv")
print(f"Phase 2 (Universal): {len(df_univ)} rows (model x draw)")

# --- Phase 3: Representational ---
# Logit lens: model x draw x band
df_logit = pd.read_csv(
    PHASE3_DIR
    / "outputs"
    / "logit_lens"
    / "base"
    / "analysis"
    / "03_master_logit_lens.csv"
)
df_logit = df_logit[df_logit["band"].isin(CONDITIONS)].copy()
print(f"Phase 3 (Logit Lens): {len(df_logit)} rows (model x draw x band)")

# Residual stream: model x draw
df_resid = pd.read_csv(
    PHASE3_DIR
    / "outputs"
    / "residual_stream"
    / "base"
    / "analysis"
    / "02_master_residual.csv"
)
print(f"Phase 3 (Residual): {len(df_resid)} rows (model x draw)")

# Info-theoretic: model x draw
df_info = pd.read_csv(
    PHASE3_DIR
    / "outputs"
    / "info_theoretic"
    / "base"
    / "analysis"
    / "06_master_info_theoretic.csv"
)
print(f"Phase 3 (Info-Theoretic): {len(df_info)} rows (model x draw)")

# Pairwise transfer: model x band_pair
df_pairwise = pd.read_csv(
    PHASE3_DIR
    / "outputs"
    / "transfer"
    / "analysis"
    / "07_pairwise_transfer_metrics.csv"
)
print(f"Phase 3 (Pairwise Transfer): {len(df_pairwise)} rows (model x band_pair)")

# --- Phase 5: Targeted (model-level) ---
p5_path = PHASE5_DIR / "outputs" / "analysis" / "scaling_summary.csv"
df_scaling = pd.read_csv(p5_path) if p5_path.exists() else pd.DataFrame()
print(f"Phase 5 (Scaling): {len(df_scaling)} rows (model-level)")

Phase 1 (Functional): 60 rows, columns: ['model', 'band', 'draw', 'threshold', 'n_edges', 'total_edges', 'size_fraction', 'base_accuracy', 'base_top5_accuracy', 'base_top10_accuracy', 'base_mean_correct_prob', 'circuit_accuracy', 'circuit_top5_accuracy', 'circuit_top10_accuracy', 'circuit_mean_correct_prob', 'circuit_kl_div', 'ablation_accuracy', 'ablation_top5_accuracy', 'ablation_top10_accuracy', 'ablation_mean_correct_prob', 'ablation_kl_div', 'retention_ratio', 'completeness', 'top1_top5_gap', 'frequency_rank', 'necessity_pass', 'training_time_seconds']
Phase 2 (Structural): 60 rows
Phase 2 (Universal): 20 rows (model x draw)
Phase 3 (Logit Lens): 60 rows (model x draw x band)
Phase 3 (Residual): 15 rows (model x draw)
Phase 3 (Info-Theoretic): 15 rows (model x draw)


Phase 3 (Pairwise Transfer): 50 rows (model x band_pair)
Phase 5 (Scaling): 4 rows (model-level)


## 2. Per-Circuit Table (model x condition x replication)

In [3]:
JOIN_KEYS = ["model", "band", "draw"]
JOIN_KEYS_MD = ["model", "draw"]  # for modelxdraw level data

# --- Select columns from each phase ---

# Phase 1: Functional metrics
func_cols = JOIN_KEYS + [
    "n_edges",
    "total_edges",
    "size_fraction",
    "base_accuracy",
    "circuit_accuracy",
    "circuit_kl_div",
    "retention_ratio",
    "completeness",
    "frequency_rank",
]
df_f = df_func[[c for c in func_cols if c in df_func.columns]].copy()

# Phase 2: Structural metrics
struct_cols = JOIN_KEYS + [
    "edge_fraction",
    "skip_fraction",
    "attn_fraction",
    "mlp_fraction",
    "resid_fraction",
    "head_participation_rate",
    "active_heads",
    "mean_edges_per_head",
]
df_s = df_struct[[c for c in struct_cols if c in df_struct.columns]].copy()

# Phase 2: Universal edges (model x draw -> broadcast to all bands)
univ_cols = JOIN_KEYS_MD + ["n_universal", "universal_fraction"]
df_u = df_univ[[c for c in univ_cols if c in df_univ.columns]].copy()

# Phase 3: Logit lens (model x draw x band)
logit_cols = JOIN_KEYS + ["convergence_layer", "frac_converged", "final_prob_correct"]
df_l = df_logit[[c for c in logit_cols if c in df_logit.columns]].copy()

# Phase 3: Residual stream (model x draw -> broadcast)
resid_cols = JOIN_KEYS_MD + [
    "peak_probe_accuracy",
    "peak_probe_layer",
    "peak_separation_ratio",
]
df_r = df_resid[[c for c in resid_cols if c in df_resid.columns]].copy()

# Phase 3: Info-theoretic (model x draw -> broadcast)
info_cols = JOIN_KEYS_MD + ["peak_mi_probe", "peak_mi_probe_layer", "peak_efficiency"]
df_i = df_info[[c for c in info_cols if c in df_info.columns]].copy()

# --- Join ---
# Start with functional (60 rows)
unified = df_f.copy()

# Merge structural (model x band x draw)
unified = unified.merge(df_s, on=JOIN_KEYS, how="left", suffixes=("", "_struct"))

# Merge universal edges (model x draw -> broadcasts to all bands)
unified = unified.merge(df_u, on=JOIN_KEYS_MD, how="left")

# Merge logit lens (model x draw x band)
unified = unified.merge(df_l, on=JOIN_KEYS, how="left")

# Merge residual stream (model x draw -> broadcasts)
unified = unified.merge(df_r, on=JOIN_KEYS_MD, how="left")

# Merge info-theoretic (model x draw -> broadcasts)
unified = unified.merge(df_i, on=JOIN_KEYS_MD, how="left")

# Add perspective labels for downstream grouping
unified["model_capacity"] = unified["model"].map(
    {
        m: c
        for m, c in zip(
            MODELS,
            [v for v in MODEL_CAPACITY.values()]
            if isinstance(MODEL_CAPACITY, dict)
            else MODEL_CAPACITY,
        )
    }
)
# Fallback: try direct mapping
if unified["model_capacity"].isna().all():
    cap_map = {
        "pythia-70m": 70,
        "pythia-160m": 160,
        "pythia-410m": 410,
        "pythia-1b": 1000,
        "pythia-1.4b": 1400,
    }
    unified["model_capacity"] = unified["model"].map(cap_map)

print(f"Unified per-circuit table: {unified.shape}")
print(f"Columns ({len(unified.columns)}): {list(unified.columns)}")
print(f"\nNull counts:")
nulls = unified.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else "  No nulls!")
unified.head(3)

Unified per-circuit table: (60, 32)
Columns (32): ['model', 'band', 'draw', 'n_edges', 'total_edges', 'size_fraction', 'base_accuracy', 'circuit_accuracy', 'circuit_kl_div', 'retention_ratio', 'completeness', 'frequency_rank', 'edge_fraction', 'skip_fraction', 'attn_fraction', 'mlp_fraction', 'resid_fraction', 'head_participation_rate', 'active_heads', 'mean_edges_per_head', 'n_universal', 'universal_fraction', 'convergence_layer', 'frac_converged', 'final_prob_correct', 'peak_probe_accuracy', 'peak_probe_layer', 'peak_separation_ratio', 'peak_mi_probe', 'peak_mi_probe_layer', 'peak_efficiency', 'model_capacity']

Null counts:
  No nulls!


,model,band,draw,n_edges,total_edges,size_fraction,base_accuracy,circuit_accuracy,circuit_kl_div,retention_ratio,...,convergence_layer,frac_converged,final_prob_correct,peak_probe_accuracy,peak_probe_layer,peak_separation_ratio,peak_mi_probe,peak_mi_probe_layer,peak_efficiency,model_capacity
0,pythia-70m,low,draw_1,380,1324,0.287009,0.284444,0.208889,0.248796,0.734375,...,4.675556,1.0,0.100825,0.610667,3,11.665066,0.967410,3,0.001482,70
1,pythia-70m,low,draw_2,396,1324,0.299094,0.360000,0.311111,0.224973,0.864198,...,4.728889,1.0,0.136085,0.581333,2,10.124512,0.935365,2,0.001423,70
2,pythia-70m,low,draw_3,391,1324,0.295317,0.257778,0.208889,0.214465,0.810345,...,4.768889,1.0,0.099103,0.588444,4,14.383308,0.907790,4,0.001344,70


In [4]:
# Save per-circuit table
unified.to_csv(ANALYSIS_DIR / "unified_per_circuit.csv", index=False)
print(
    f"Saved: unified_per_circuit.csv ({unified.shape[0]} rows, {unified.shape[1]} columns)"
)

Saved: unified_per_circuit.csv (60 rows, 32 columns)


## 3. Pairwise Table (model x condition_pair)

In [5]:
# Start from Phase 3 pairwise transfer metrics
pairwise = df_pairwise.copy()

# Enrich with Phase 2 Jaccard data if not already present
if "jaccard" not in pairwise.columns:
    df_jacc = pd.read_csv(PHASE2_DIR / "outputs" / "analysis" / "band_jaccard.csv")
    pairwise = pairwise.merge(
        df_jacc[["model", "band_1", "band_2", "mean_jaccard"]],
        on=["model", "band_1", "band_2"],
        how="left",
    )
    pairwise.rename(columns={"mean_jaccard": "jaccard"}, inplace=True)

# Save
pairwise.to_csv(ANALYSIS_DIR / "unified_pairwise.csv", index=False)
print(
    f"Saved: unified_pairwise.csv ({pairwise.shape[0]} rows, {pairwise.shape[1]} columns)"
)
print(f"Columns: {list(pairwise.columns)}")
pairwise.head(3)

Saved: unified_pairwise.csv (50 rows, 13 columns)
Columns: ['model', 'band_1', 'band_2', 'model_capacity', 'freq_distance', 'transfer_12', 'transfer_21', 'transfer_mean', 'transfer_asymmetry', 'jaccard', 'embedding_cka', 'convergence_diff', 'peak_separation']


,model,band_1,band_2,model_capacity,freq_distance,transfer_12,transfer_21,transfer_mean,transfer_asymmetry,jaccard,embedding_cka,convergence_diff,peak_separation
0,pythia-70m,low,medium,70,1.0,0.322963,0.251852,0.287407,0.071111,0.790890,0.474499,0.093333,14.383308
1,pythia-70m,low,high,70,2.0,0.358519,0.269630,0.314074,0.088889,0.770530,0.469160,0.219259,14.383308
2,pythia-70m,low,very_high,70,3.0,0.468148,0.231111,0.349630,0.237037,0.724777,0.470316,0.383704,14.383308


## 4. Model-Level Table

In [6]:
# Aggregate per-circuit table by model
numeric_cols = unified.select_dtypes(include=[np.number]).columns.tolist()
# Remove join/id columns from aggregation
agg_cols = [c for c in numeric_cols if c not in ["frequency_rank", "model_capacity"]]

model_agg = unified.groupby("model")[agg_cols].agg(["mean", "std"]).reset_index()
# Flatten multi-level columns
model_agg.columns = ["model"] + [f"{c}_{stat}" for c, stat in model_agg.columns[1:]]

# Add model capacity
cap_map = {
    "pythia-70m": 70,
    "pythia-160m": 160,
    "pythia-410m": 410,
    "pythia-1b": 1000,
    "pythia-1.4b": 1400,
}
model_agg.insert(1, "model_capacity", model_agg["model"].map(cap_map))

# Enrich with Phase 5 scaling summary if available
if not df_scaling.empty:
    p5_cols = ["model"] + [
        c for c in df_scaling.columns if c not in model_agg.columns and c != "params"
    ]
    p5_available = [c for c in p5_cols if c in df_scaling.columns]
    model_agg = model_agg.merge(df_scaling[p5_available], on="model", how="left")

# Save
model_agg.to_csv(ANALYSIS_DIR / "unified_model_level.csv", index=False)
print(
    f"Saved: unified_model_level.csv ({model_agg.shape[0]} rows, {model_agg.shape[1]} columns)"
)
print(f"\nColumns: {list(model_agg.columns)}")
model_agg

Saved: unified_model_level.csv (5 rows, 74 columns)

Columns: ['model', 'model_capacity', 'n_edges_mean', 'n_edges_std', 'total_edges_mean', 'total_edges_std', 'size_fraction_mean', 'size_fraction_std', 'base_accuracy_mean', 'base_accuracy_std', 'circuit_accuracy_mean', 'circuit_accuracy_std', 'circuit_kl_div_mean', 'circuit_kl_div_std', 'retention_ratio_mean', 'retention_ratio_std', 'completeness_mean', 'completeness_std', 'edge_fraction_mean', 'edge_fraction_std', 'skip_fraction_mean', 'skip_fraction_std', 'attn_fraction_mean', 'attn_fraction_std', 'mlp_fraction_mean', 'mlp_fraction_std', 'resid_fraction_mean', 'resid_fraction_std', 'head_participation_rate_mean', 'head_participation_rate_std', 'active_heads_mean', 'active_heads_std', 'mean_edges_per_head_mean', 'mean_edges_per_head_std', 'n_universal_mean', 'n_universal_std', 'universal_fraction_mean', 'universal_fraction_std', 'convergence_layer_mean', 'convergence_layer_std', 'frac_converged_mean', 'frac_converged_std', 'final_pro

,model,model_capacity,n_edges_mean,n_edges_std,total_edges_mean,total_edges_std,size_fraction_mean,size_fraction_std,base_accuracy_mean,base_accuracy_std,...,cross_band_boost,transfer_efficiency,random_boost,critical_k_mean,k3_recovery,cross_draw_transfer,frac_draw_exclusive,jaccard_within,jaccard_between,jaccard_gap
0,pythia-1.4b,1400,2307.750000,240.146593,80581.0,0.0,0.028639,0.002980,0.978148,0.016786,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,pythia-160m,160,1415.083333,66.703494,11467.0,0.0,0.123405,0.005817,0.953704,0.016353,...,0.281630,0.926413,0.034015,3.0,1.000278,0.998361,0.327891,0.589080,0.556812,0.032268
2,pythia-1b,1000,925.166667,63.977032,10009.0,0.0,0.092433,0.006392,0.986667,0.008889,...,0.484593,0.944284,0.034726,3.0,1.027741,1.003541,0.421317,0.477929,0.465130,0.012798
3,pythia-410m,410,3632.833333,304.680110,80581.0,0.0,0.045083,0.003781,0.988148,0.006655,...,0.434074,0.964450,0.016711,3.6,1.002385,0.998952,0.460191,0.446278,0.430386,0.015891
4,pythia-70m,70,405.666667,15.293096,1324.0,0.0,0.306395,0.011551,0.468148,0.146260,...,0.099481,0.814927,0.010785,2.8,0.962040,0.986929,0.161756,0.794989,0.762580,0.032409


## 5. Metric Catalog

In [7]:
# Build metric catalog: what comes from where
catalog_rows = []

func_metrics = {
    "n_edges": "Number of edges in circuit",
    "total_edges": "Total possible edges in model",
    "size_fraction": "Circuit size as fraction of total",
    "base_accuracy": "Full model accuracy",
    "circuit_accuracy": "Circuit-only accuracy",
    "circuit_kl_div": "KL divergence between circuit and full model",
    "retention_ratio": "Circuit accuracy / base accuracy",
    "completeness": "Ablation completeness score",
}
for m, desc in func_metrics.items():
    catalog_rows.append(
        {
            "metric": m,
            "perspective": "functional",
            "phase": 1,
            "source_file": "full_circuit_data.csv",
            "granularity": "modelxconditionxreplication",
            "description": desc,
        }
    )

struct_metrics = {
    "edge_fraction": "Fraction of total possible edges in circuit",
    "skip_fraction": "Fraction of circuit edges that are skip connections",
    "attn_fraction": "Fraction of circuit edges in attention",
    "mlp_fraction": "Fraction of circuit edges in MLP",
    "resid_fraction": "Fraction of circuit edges in residual stream",
    "head_participation_rate": "Fraction of attention heads with >=1 circuit edge",
    "active_heads": "Number of attention heads with >=1 circuit edge",
    "mean_edges_per_head": "Mean circuit edges per active attention head",
    "n_universal": "Number of edges present in all conditions (modelxreplication level)",
    "universal_fraction": "Fraction of circuit edges that are universal",
}
for m, desc in struct_metrics.items():
    gran = (
        "modelxreplication"
        if m in ("n_universal", "universal_fraction")
        else "modelxconditionxreplication"
    )
    catalog_rows.append(
        {
            "metric": m,
            "perspective": "structural",
            "phase": 2,
            "source_file": "universal_edges.csv"
            if "universal" in m
            else "full_structure_data.csv",
            "granularity": gran,
            "description": desc,
        }
    )

repr_metrics = {
    "convergence_layer": "Layer where logit lens probability reaches 90% of final",
    "frac_converged": "Fraction of examples that converge",
    "final_prob_correct": "Final layer probability of correct token",
    "peak_probe_accuracy": "Peak linear probe accuracy across layers",
    "peak_probe_layer": "Layer with peak probe accuracy",
    "peak_separation_ratio": "Peak between/within cluster distance ratio",
    "peak_mi_probe": "Peak probe-based mutual information across layers",
    "peak_mi_probe_layer": "Layer with peak probe MI",
    "peak_efficiency": "Peak information coding efficiency",
}
for m, desc in repr_metrics.items():
    if (
        m.startswith("convergence")
        or m.startswith("frac_conv")
        or m.startswith("final_prob")
    ):
        src = "03_master_logit_lens.csv"
        gran = "modelxconditionxreplication"
    elif m.startswith("peak_probe") or m.startswith("peak_sep"):
        src = "02_master_residual.csv"
        gran = "modelxreplication"
    else:
        src = "06_master_info_theoretic.csv"
        gran = "modelxreplication"
    catalog_rows.append(
        {
            "metric": m,
            "perspective": "representational",
            "phase": 3,
            "source_file": src,
            "granularity": gran,
            "description": desc,
        }
    )

df_catalog = pd.DataFrame(catalog_rows)
df_catalog.to_csv(ANALYSIS_DIR / "metric_catalog.csv", index=False)
print(f"Saved: metric_catalog.csv ({len(df_catalog)} metrics)")
print(f"\nMetrics by perspective:")
print(df_catalog.groupby("perspective").size())
print()
df_catalog

Saved: metric_catalog.csv (27 metrics)

Metrics by perspective:
perspective
functional           8
representational     9
structural          10
dtype: int64



,metric,perspective,phase,source_file,granularity,description
0,n_edges,functional,1,full_circuit_data.csv,modelxconditionxreplication,Number of edges in circuit
1,total_edges,functional,1,full_circuit_data.csv,modelxconditionxreplication,Total possible edges in model
2,size_fraction,functional,1,full_circuit_data.csv,modelxconditionxreplication,Circuit size as fraction of total
3,base_accuracy,functional,1,full_circuit_data.csv,modelxconditionxreplication,Full model accuracy
4,circuit_accuracy,functional,1,full_circuit_data.csv,modelxconditionxreplication,Circuit-only accuracy
5,circuit_kl_div,functional,1,full_circuit_data.csv,modelxconditionxreplication,KL divergence between circuit and full model
6,retention_ratio,functional,1,full_circuit_data.csv,modelxconditionxreplication,Circuit accuracy / base accuracy
7,completeness,functional,1,full_circuit_data.csv,modelxconditionxreplication,Ablation completeness score
8,edge_fraction,structural,2,full_structure_data.csv,modelxconditionxreplication,Fraction of total possible edges in circuit
9,skip_fraction,structural,2,full_structure_data.csv,modelxconditionxreplication,Fraction of circuit edges that are skip connec...


## 6. Summary & Quality Check

In [8]:
print("=" * 80)
print("PHASE 4: INTEGRATION 1: UNIFIED DATA TABLE")
print("=" * 80)

print(f"\n--- Tables Created ---")
for name, df in [
    ("unified_per_circuit.csv", unified),
    ("unified_pairwise.csv", pairwise),
    ("unified_model_level.csv", model_agg),
    ("metric_catalog.csv", df_catalog),
]:
    path = ANALYSIS_DIR / name
    size_kb = path.stat().st_size / 1024 if path.exists() else 0
    print(f"  {name}: {df.shape[0]} rows x {df.shape[1]} columns ({size_kb:.1f} KB)")

print(f"\n--- Per-Circuit Table Coverage ---")
print(f"  Models: {sorted(unified['model'].unique())}")
print(f"  Conditions: {sorted(unified['band'].unique())}")
print(f"  Replications: {sorted(unified['draw'].unique())}")
print(
    f"  Total rows: {len(unified)} (expected {len(MODELS) * len(CONDITIONS) * len(REPLICATIONS)})"
)

print(f"\n--- Perspective Coverage ---")
perspective_cols = {
    "Functional": [c for c in unified.columns if c in func_metrics],
    "Structural": [c for c in unified.columns if c in struct_metrics],
    "Representational": [c for c in unified.columns if c in repr_metrics],
}
for p, cols in perspective_cols.items():
    n_null = unified[cols].isnull().sum().sum()
    print(f"  {p}: {len(cols)} metrics, {n_null} null values")

print(f"\n--- Pairwise Table ---")
print(f"  {len(pairwise)} condition pairs across {pairwise['model'].nunique()} models")

print(f"\n--- Model-Level Table ---")
print(f"  {len(model_agg)} models, {len(model_agg.columns)} columns")

print("\nDone.")

PHASE 4: INTEGRATION 1: UNIFIED DATA TABLE

--- Tables Created ---
  unified_per_circuit.csv: 60 rows x 32 columns (24.6 KB)
  unified_pairwise.csv: 50 rows x 13 columns (8.6 KB)
  unified_model_level.csv: 5 rows x 74 columns (7.4 KB)
  metric_catalog.csv: 27 rows x 6 columns (3.2 KB)

--- Per-Circuit Table Coverage ---
  Models: ['pythia-1.4b', 'pythia-160m', 'pythia-1b', 'pythia-410m', 'pythia-70m']
  Conditions: ['high', 'low', 'medium', 'very_high']
  Replications: ['draw_1', 'draw_2', 'draw_3']
  Total rows: 60 (expected 60)

--- Perspective Coverage ---
  Functional: 8 metrics, 0 null values


  Structural: 10 metrics, 0 null values
  Representational: 9 metrics, 0 null values

--- Pairwise Table ---
  50 condition pairs across 5 models

--- Model-Level Table ---
  5 models, 74 columns

Done.
